# P5 — Granite na própria LinuxONE

Este notebook roda **na VM da equipe** (s390x). Ele:

1. descobre o que a VM oferece para rodar o arquivo `granite-4.0-h-350m-base-be.f16.gguf`;
2. confere se o servidor do modelo (llama.cpp) está no ar;
3. gera explicações de alertas com esse Granite e mostra o que a validação aceitou ou recusou.

Sobre o arquivo: GGUF é o formato do **llama.cpp**; `-be` = big-endian (obrigatório no s390x);
`f16` = pesos em 16 bits (~650 MB); `h-350m` = Granite 4.0 híbrido de 350 milhões de parâmetros;
`base` = **não** é ajustado para seguir instruções. Por isso o prompt é por exemplos (few-shot),
e todo texto passa pela mesma validação: se inventar algo, volta o texto padrão.

**A VM é compartilhada (3 vCPUs para a equipe toda):** use 1 thread, desligue o servidor e o
kernel (aba *Running*) quando terminar. Os alertas aqui são **mock** (inventados ou do detector
de teste); nada disso é detecção de vazamento.

In [ ]:
import os
import sys
import time
from pathlib import Path

# Sobe a partir da pasta do notebook até o clone (ex.: ~/shared/civicwater/users/<nome>/repo).
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(REPO / "src"))

for v in ["OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS"]:
    os.environ[v] = "1"

# Servidor do modelo: troque a porta se outra pessoa já usa a 8080.
os.environ.setdefault("LOCAL_LLM_URL", "http://127.0.0.1:8080")
os.environ.setdefault("LOCAL_LLM_TIMEOUT_S", "120")

print("repo:", REPO)
print("python do kernel:", sys.executable, sys.version.split()[0])

## 1. O que esta VM tem (só leitura)

Copie a saída desta célula para quem for ajudar: ela diz qual caminho seguir na seção 2.

In [ ]:
import glob
import importlib.util
import platform
import shutil
import struct

home = Path.home()
print("arquitetura:", platform.machine(), "| CPUs:", os.cpu_count())
try:
    meminfo = Path("/proc/meminfo").read_text().splitlines()
    print(*[l for l in meminfo if l.startswith(("MemTotal", "MemAvailable"))], sep="\n")
    cpuinfo = Path("/proc/cpuinfo").read_text()
    print("NNPA (acelerador de IA Telum) visível:", " nnpa" in cpuinfo or "nnpa " in cpuinfo)
except OSError:
    pass

print("\nprogramas:")
for b in ["llama-server", "llama-cli", "llama-run", "ollama", "podman", "docker", "gcc", "cmake", "uv"]:
    print(f"  {b:13} {shutil.which(b)}")

print("\npacotes Python neste kernel:")
for m in ["llama_cpp", "pydantic", "fastapi", "pytest", "torch", "transformers", "onnxruntime", "snapml"]:
    print(f"  {m:13} {'sim' if importlib.util.find_spec(m) else '-'}")

print("\nprogramas llama.cpp fora do PATH:")
for pattern in ["/opt/*llama*", "/opt/*/bin/llama-*", "/opt/*/*/bin/llama-*", "/usr/local/bin/llama-*",
                str(home / "*llama*"), str(home / "*/bin/llama-*"), str(home / "*/build/bin/llama-*")]:
    for hit in glob.glob(pattern)[:5]:
        print("  ", hit)

print("\narquivos .gguf:")
ggufs = sorted(set(glob.glob(str(home / "*.gguf")) + glob.glob(str(home / "*/*.gguf"))))
for p in ggufs:
    with open(p, "rb") as f:
        head = f.read(8)
    if head[:4] != b"GGUF":
        print("  ", p, "-> não parece GGUF")
        continue
    le, be = struct.unpack("<I", head[4:]), struct.unpack(">I", head[4:])
    endian = "big-endian" if be[0] < le[0] else "little-endian"
    print(f"   {p}  ({os.path.getsize(p) / 1e6:.0f} MB, GGUF v{min(le[0], be[0])}, {endian})")

print("\nnotebooks que mencionam gguf/llama/granite (exemplos prontos da VM?):")
for nb in glob.glob(str(home / "*.ipynb")) + glob.glob(str(home / "*/*.ipynb")):
    try:
        text = Path(nb).read_text(errors="ignore").lower()
    except OSError:
        continue
    if any(w in text for w in ("gguf", "llama", "granite")):
        print("  ", nb)

## 2. Subir o servidor do modelo

Abra um terminal no Jupyter (**New → Terminal**) e use **uma** das opções, conforme a seção 1:

**A. Existe `llama-server`** (no PATH ou num caminho listado acima):

```bash
llama-server -m ~/granite-4.0-h-350m-base-be.f16.gguf --host 127.0.0.1 --port 8080 -t 1 -c 4096
```

Ao iniciar, ele imprime as capacidades da CPU usadas (procure `VXE`, e `ZDNN`/`NNPA` se o build
tiver aceleração). Deixe o terminal aberto; `Ctrl+C` desliga.

**B. Existe o pacote `llama_cpp` (llama-cpp-python com extra `server`)**:

```bash
python -m llama_cpp.server --model ~/granite-4.0-h-350m-base-be.f16.gguf --host 127.0.0.1 --port 8080 --n_threads 1
```

**C. Nenhum dos dois:** a VM não tem `gcc`/`cmake` para compilar o llama.cpp e não há binário
oficial para s390x. Pergunte a quem colocou o modelo na VM (organização do Datathon / dono da VM)
qual runtime usar: um `llama-server` já compilado, um contêiner, ou a liberação de
`sudo apt install gcc g++ cmake libopenblas-dev` para compilar (Q-PLAT-02 em
`docs/questions-for-team.md`).

Porta ocupada por outra pessoa? Use outra (ex.: 8081) e ajuste `LOCAL_LLM_URL` na primeira célula.

In [ ]:
import json
import urllib.request

url = os.environ["LOCAL_LLM_URL"]
for path in ("/health", "/v1/models"):
    try:
        with urllib.request.urlopen(url + path, timeout=5) as r:
            print(path, r.status, r.read(300).decode(errors="replace"))
    except Exception as exc:
        print(path, "sem resposta:", exc)

## 3. Explicar um alerta com o Granite local

`explain()` manda ao modelo um prompt com dois exemplos inventados e os fatos do alerta.
O texto só é aceito se **todo** sensor, região, número, data e horário estiver no alerta;
senão volta o texto padrão, com o motivo em `fallback_reason`.

In [ ]:
from sentinel.narrative import LocalGraniteNarrator, explain
from sentinel.narrative.local import build_completion_prompt
from sentinel.narrative.mock import mock_alert

granite = LocalGraniteNarrator.from_env()
alerta = mock_alert()  # inventado: região R4, CUSUM, 3 sensores

print(build_completion_prompt(alerta)[-700:], "\n")
t = time.perf_counter()
e = explain(alerta, granite)
print(f"tempo: {time.perf_counter() - t:.1f} s | fonte: {e.source} | fallback: {e.fallback_reason}")
print("validação:", e.check)
print("\n" + e.text)

## 4. Mesmo teste com um alerta do pipeline (detector mock do backend)

In [ ]:
from datetime import UTC, datetime

from sentinel.audit import HashChainAuditSink
from sentinel.config import Settings
from sentinel.main import build_pipeline
from sentinel.schemas.windows import SensorWindow
from sentinel.sensors import SensorRegistry
from sentinel.synthetic import generate

registry = SensorRegistry.load(REPO / "config" / "sensors.provisional.json")
pipeline = build_pipeline(Settings(), registry=registry, audit_sink=HashChainAuditSink())
for w in generate(datetime(2019, 3, 14, tzinfo=UTC), 288, registry, seed=7, noise_m=0.05,
                  leak_at=200, leak_sensors=registry.pressure[:3]):
    pipeline.ingest(SensorWindow.model_validate(w))

alerta_pipeline = pipeline.alerts.query(limit=1, offset=0)[0][0]
t = time.perf_counter()
e = explain(alerta_pipeline, granite)
print(f"tempo: {time.perf_counter() - t:.1f} s | fonte: {e.source} | fallback: {e.fallback_reason}")
print("validação:", e.check)
print("\n" + e.text)

## 5. Como ler o resultado

- `fonte: llm` → o texto do Granite passou na validação e seria mostrado ao operador.
- `fonte: template` com `narrative_rejected` → o modelo inventou ou copiou algo (veja `validação`);
  o operador vê o texto padrão. Com um modelo *base* de 350M isso pode ser frequente.
- `narrator_error` → servidor fora do ar, porta errada ou tempo esgotado (`LOCAL_LLM_TIMEOUT_S`).

Anote o **tempo** por explicação: ele decide se a explicação pode ser gerada na hora em que o
operador abre o alerta ou se deve ser gerada em segundo plano. Se a taxa de aceitação for baixa,
as alternativas são um Granite *instruct* convertido para big-endian ou o `ibm/granite-4-h-small`
no watsonx (`WatsonxNarrator`).

Ao terminar: `Ctrl+C` no terminal do servidor e *Running → Shut Down* neste kernel.